# AlexNet — Entrenamiento

In [ ]:
# CELDA 1 — Montar Drive y descomprimir
from google.colab import drive
drive.mount('/content/drive')

import subprocess
result = subprocess.run([
    'unzip', '-q',
    '/content/drive/MyDrive/Datasets/Dataset_P2_DL.zip',
    '-d', '/content/data'
])
print('Descompresión lista')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Descompresión lista


In [ ]:
# CELDA 2 — Limpiar archivos corruptos
from PIL import Image
import os

corrupted = []
for root, _, files in os.walk('/content/data'):
    for fname in files:
        fpath = os.path.join(root, fname)
        try:
            with Image.open(fpath) as img:
                img.verify()
        except Exception:
            corrupted.append(fpath)

print(f'Archivos corruptos: {len(corrupted)}')
for f in corrupted:
    print(f)
    os.remove(f)
print('Eliminados')

Archivos corruptos: 0
Eliminados


### Baseline

In [ ]:
# CELDA 3 — Variantes 1: Arquitectura AlexNet A1: + BatchNorm2d tras cada conv
import torch
from torch.nn import Conv2d, BatchNorm2d, ReLU, MaxPool2d, Linear, Module, Flatten

class AlexNet(Module):
    def __init__(self, in_dim, n_classes):
        super(AlexNet, self).__init__()
        # Capas convolucionales + BN
        self.conv1 = Conv2d(in_dim, 96,  kernel_size=11, stride=4, padding=0)
        self.bn1   = BatchNorm2d(96)
        self.conv2 = Conv2d(96,  256, kernel_size=5,  stride=1, padding=2)
        self.bn2   = BatchNorm2d(256)
        self.conv3 = Conv2d(256, 384, kernel_size=3,  stride=1, padding=1)
        self.bn3   = BatchNorm2d(384)
        self.conv4 = Conv2d(384, 384, kernel_size=3,  stride=1, padding=1)
        self.bn4   = BatchNorm2d(384)
        self.conv5 = Conv2d(384, 256, kernel_size=3,  stride=1, padding=1)
        self.bn5   = BatchNorm2d(256)

        # Pooling y activación
        self.pool    = MaxPool2d(kernel_size=3, stride=2)
        self.relu    = ReLU(inplace=True)
        self.flatten = Flatten()

        # Capas lineales (sin cambios)
        self.fc1 = Linear(5 * 5 * 256, 4096)
        self.fc2 = Linear(4096, 4096)
        self.fc3 = Linear(4096, n_classes)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        x = self.relu(self.bn3(self.conv3(x)))
        x = self.relu(self.bn4(self.conv4(x)))
        x = self.relu(self.bn5(self.conv5(x)))
        x = self.pool(x)
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

print('Arquitectura A1 (AlexNet + BN) definida')

Arquitectura A1 (AlexNet + BN) definida


In [ ]:
# CELDA 3 — Variante 2 Arquitectura A2: AlexNet + Dropout en FC
import torch
from torch.nn import Conv2d, ReLU, MaxPool2d, Linear, Dropout, Module, Flatten

class AlexNet(Module):
    def __init__(self, in_dim, n_classes):
        super(AlexNet, self).__init__()
        # Capas convolucionales (sin cambios)
        self.conv1 = Conv2d(in_dim, 96,  kernel_size=11, stride=4, padding=0)
        self.conv2 = Conv2d(96,  256, kernel_size=5,  stride=1, padding=2)
        self.conv3 = Conv2d(256, 384, kernel_size=3,  stride=1, padding=1)
        self.conv4 = Conv2d(384, 384, kernel_size=3,  stride=1, padding=1)
        self.conv5 = Conv2d(384, 256, kernel_size=3,  stride=1, padding=1)

        # Pooling y activación
        self.pool    = MaxPool2d(kernel_size=3, stride=2)
        self.relu    = ReLU(inplace=True)
        self.flatten = Flatten()
        self.dropout = Dropout(p=0.5)  # ← único añadido

        # Capas lineales (sin cambios en tamaño)
        self.fc1 = Linear(5 * 5 * 256, 4096)
        self.fc2 = Linear(4096, 4096)
        self.fc3 = Linear(4096, n_classes)

    def forward(self, x):
        # Bloque conv (igual que baseline)
        x = self.relu(self.conv1(x))
        x = self.pool(x)
        x = self.relu(self.conv2(x))
        x = self.pool(x)
        x = self.relu(self.conv3(x))
        x = self.relu(self.conv4(x))
        x = self.relu(self.conv5(x))
        x = self.pool(x)
        x = self.flatten(x)
        # FC con Dropout
        x = self.dropout(self.relu(self.fc1(x)))  # ← dropout tras fc1
        x = self.dropout(self.relu(self.fc2(x)))  # ← dropout tras fc2
        x = self.fc3(x)
        return x

print('Arquitectura A2 (AlexNet + Dropout) definida')

Arquitectura A2 (AlexNet + Dropout) definida


In [ ]:
# CELDA 3 — Variante 3 reducir fc1 y fc2 de 4096 → 1024 Arquitectura A3: AlexNet + FC reducidas
import torch
from torch.nn import Conv2d, ReLU, MaxPool2d, Linear, Module, Flatten

class AlexNet(Module):
    def __init__(self, in_dim, n_classes):
        super(AlexNet, self).__init__()
        # Capas convolucionales (sin cambios)
        self.conv1 = Conv2d(in_dim, 96,  kernel_size=11, stride=4, padding=0)
        self.conv2 = Conv2d(96,  256, kernel_size=5,  stride=1, padding=2)
        self.conv3 = Conv2d(256, 384, kernel_size=3,  stride=1, padding=1)
        self.conv4 = Conv2d(384, 384, kernel_size=3,  stride=1, padding=1)
        self.conv5 = Conv2d(384, 256, kernel_size=3,  stride=1, padding=1)

        # Pooling y activación
        self.pool    = MaxPool2d(kernel_size=3, stride=2)
        self.relu    = ReLU(inplace=True)
        self.flatten = Flatten()

        # FC reducidas 4096 → 1024
        self.fc1 = Linear(5 * 5 * 256, 1024)  # ← cambio
        self.fc2 = Linear(1024, 1024)          # ← cambio
        self.fc3 = Linear(1024, n_classes)     # ← cambio

    def forward(self, x):
        # Bloque conv (igual que baseline)
        x = self.relu(self.conv1(x))
        x = self.pool(x)
        x = self.relu(self.conv2(x))
        x = self.pool(x)
        x = self.relu(self.conv3(x))
        x = self.relu(self.conv4(x))
        x = self.relu(self.conv5(x))
        x = self.pool(x)
        x = self.flatten(x)
        # FC reducidas (sin dropout, sin BN)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

print('Arquitectura A3 (AlexNet + FC reducidas) definida')

Arquitectura A3 (AlexNet + FC reducidas) definida


In [ ]:
# CELDA 3 — Variante 4 mantiene BN para conv, FC 1024 y +AdamW Arquitectura A4: AlexNet + BN + FC reducidas
import torch
from torch.nn import Conv2d, BatchNorm2d, ReLU, MaxPool2d, Linear, Module, Flatten

class AlexNet(Module):
    def __init__(self, in_dim, n_classes):
        super(AlexNet, self).__init__()
        # Capas convolucionales + BN (de A1)
        self.conv1 = Conv2d(in_dim, 96,  kernel_size=11, stride=4, padding=0)
        self.bn1   = BatchNorm2d(96)
        self.conv2 = Conv2d(96,  256, kernel_size=5,  stride=1, padding=2)
        self.bn2   = BatchNorm2d(256)
        self.conv3 = Conv2d(256, 384, kernel_size=3,  stride=1, padding=1)
        self.bn3   = BatchNorm2d(384)
        self.conv4 = Conv2d(384, 384, kernel_size=3,  stride=1, padding=1)
        self.bn4   = BatchNorm2d(384)
        self.conv5 = Conv2d(384, 256, kernel_size=3,  stride=1, padding=1)
        self.bn5   = BatchNorm2d(256)

        # Pooling y activación
        self.pool    = MaxPool2d(kernel_size=3, stride=2)
        self.relu    = ReLU(inplace=True)
        self.flatten = Flatten()

        # FC reducidas (de A3)
        self.fc1 = Linear(5 * 5 * 256, 1024)
        self.fc2 = Linear(1024, 1024)
        self.fc3 = Linear(1024, n_classes)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        x = self.relu(self.bn3(self.conv3(x)))
        x = self.relu(self.bn4(self.conv4(x)))
        x = self.relu(self.bn5(self.conv5(x)))
        x = self.pool(x)
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

print('Arquitectura A4 (AlexNet + BN + FC reducidas) definida')

Arquitectura A4 (AlexNet + BN + FC reducidas) definida


In [ ]:
# CELDA 3 — Variante 0 Base Arquitectura AlexNet adaptada a 224x224
import torch
from torch.nn import Conv2d, ReLU, MaxPool2d, Linear, Module, Flatten

class AlexNet(Module):
    def __init__(self, in_dim, n_classes):
        super(AlexNet, self).__init__()
        # Capas convolucionales
        self.conv1 = Conv2d(in_dim, 96,  kernel_size=11, stride=4, padding=0)
        self.conv2 = Conv2d(96,  256, kernel_size=5,  stride=1, padding=2)
        self.conv3 = Conv2d(256, 384, kernel_size=3,  stride=1, padding=1)
        self.conv4 = Conv2d(384, 384, kernel_size=3,  stride=1, padding=1)
        self.conv5 = Conv2d(384, 256, kernel_size=3,  stride=1, padding=1)

        # Pooling y activación
        self.pool    = MaxPool2d(kernel_size=3, stride=2)
        self.relu    = ReLU(inplace=True)
        self.flatten = Flatten()

        # Capas lineales
        self.fc1 = Linear(5 * 5 * 256, 4096)
        self.fc2 = Linear(4096, 4096)
        self.fc3 = Linear(4096, n_classes)

    def forward(self, x):
        x = self.relu(self.conv1(x))   # 224→54  después de pool→26
        x = self.pool(x)
        x = self.relu(self.conv2(x))   # 26→26   después de pool→12  (padding=2 conserva)
        x = self.pool(x)
        x = self.relu(self.conv3(x))   # sin pool
        x = self.relu(self.conv4(x))   # sin pool
        x = self.relu(self.conv5(x))
        x = self.pool(x)               # →5x5
        x = self.flatten(x)            # 5*5*256 = 6400
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

print('Arquitectura definida')

Arquitectura definida


In [ ]:
# CELDA 4 — Imports, configuración, datos y loaders
import os, time, json, copy
from pathlib import Path
import random

import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report

# ── Hiperparámetros ─────────────────────────────────────────
SEED             = 42
EPOCHS           = 60
BATCH_SIZE       = 128
LR               = 0.01
MOMENTUM         = 0.9
WEIGHT_DECAY     = 5e-4
NUM_WORKERS      = 4
CHECKPOINT_EVERY = 10
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Rutas ───────────────────────────────────────────────────
DATA_PATH      = '/content/data/Data_subsampled'
OUT_DIR = Path('/content/drive/MyDrive/Datasets/outputs_alexnet_A1_FC_1024_BN_ADAMW_V4')
CHECKPOINT_DIR = OUT_DIR / 'checkpoints'
PLOTS_DIR      = OUT_DIR / 'plots'
METRICS_DIR    = OUT_DIR / 'metrics'
for p in (CHECKPOINT_DIR, PLOTS_DIR, METRICS_DIR):
    p.mkdir(parents=True, exist_ok=True)

# ── Reproducibilidad ────────────────────────────────────────
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(SEED)

# ── Media y std del dataset ──────────────────────────────────
def compute_mean_std(data_path):
    ds     = datasets.ImageFolder(data_path, transform=transforms.ToTensor())
    loader = DataLoader(ds, batch_size=128, num_workers=4, shuffle=False,
                        persistent_workers=True)
    mean = torch.zeros(3)
    std  = torch.zeros(3)
    for imgs, _ in loader:
        mean += imgs.mean(dim=[0, 2, 3])
        std  += imgs.std(dim=[0, 2, 3])
    mean /= len(loader)
    std  /= len(loader)
    print(f'Mean: {mean}  Std: {std}')
    return mean.tolist(), std.tolist()

MEAN, STD = compute_mean_std(DATA_PATH)

# ── Transforms ──────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# ── Dataset y splits ────────────────────────────────────────
dataset      = datasets.ImageFolder(root=DATA_PATH, transform=train_transform)
dataset_eval = datasets.ImageFolder(root=DATA_PATH, transform=eval_transform)

class_names = dataset.classes
n_classes   = len(class_names)
print(f'Clases: {class_names}  |  Total: {len(dataset)}  |  Device: {DEVICE}')

targets = np.array(dataset.targets)
indices = np.arange(len(dataset))

train_idx, temp_idx, y_train, y_temp = train_test_split(
    indices, targets, train_size=0.8, stratify=targets, random_state=SEED)
val_idx, test_idx, _, _ = train_test_split(
    temp_idx, y_temp, test_size=0.5, stratify=y_temp, random_state=SEED)

train_dataset = Subset(dataset,      train_idx)
val_dataset   = Subset(dataset_eval, val_idx)
test_dataset  = Subset(dataset_eval, test_idx)
print(f'train:{len(train_dataset)}  val:{len(val_dataset)}  test:{len(test_dataset)}')

loader_kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                 pin_memory=True, persistent_workers=True, prefetch_factor=2)
train_loader = DataLoader(train_dataset, shuffle=True,  **loader_kw)
val_loader   = DataLoader(val_dataset,   shuffle=False, **loader_kw)
test_loader  = DataLoader(test_dataset,  shuffle=False, **loader_kw)
print('Loaders listos')

Mean: tensor([0.3745, 0.3034, 0.4152])  Std: tensor([0.3549, 0.3974, 0.4199])
Clases: ['Disgust', 'Fear', 'Happy', 'Neutral', 'Sad']  |  Total: 30000  |  Device: cuda
train:24000  val:3000  test:3000
Loaders listos


In [ ]:
# CELDA 5 — Utilidades y funciones de entrenamiento

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

def save_checkpoint(state, filename):
    torch.save(state, str(filename))

def save_curves(history, outdir):
    epochs = list(range(1, len(history['train_loss']) + 1))
    for keys, title, fname in [
        (('train_loss','val_loss'), 'Loss',     'loss_curve.png'),
        (('train_f1',  'val_f1'),  'Macro F1', 'f1_curve.png'),
    ]:
        plt.figure()
        for k in keys: plt.plot(epochs, history[k], label=k)
        plt.xlabel('Epoch'); plt.ylabel(title)
        plt.title(f'{title} - Train/Val'); plt.legend(); plt.grid(True)
        plt.savefig(outdir / fname, dpi=150); plt.close()
    import csv
    with open(outdir / 'metrics_per_epoch.csv', 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['epoch','train_loss','val_loss','train_f1','val_f1','epoch_time_s'])
        for i in range(len(epochs)):
            w.writerow([epochs[i], history['train_loss'][i], history['val_loss'][i],
                        history['train_f1'][i], history['val_f1'][i], history['epoch_times'][i]])

def save_confusion_matrix(y_true, y_pred, classes, outpath, normalize=True):
    cm = confusion_matrix(y_true, y_pred)
    cm_plot = cm.astype('float') / (cm.sum(axis=1)[:, None] + 1e-12) if normalize else cm
    plt.figure(figsize=(8, 6))
    plt.imshow(cm_plot, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title('Matriz de confusión'); plt.colorbar()
    ticks = np.arange(len(classes))
    plt.xticks(ticks, classes, rotation=45, ha='right')
    plt.yticks(ticks, classes)
    fmt = '.2f' if normalize else 'd'
    thresh = cm_plot.max() / 2.
    for i, j in np.ndindex(cm_plot.shape):
        plt.text(j, i, format(cm_plot[i, j], fmt), ha='center',
                 color='white' if cm_plot[i, j] > thresh else 'black')
    plt.ylabel('Clase real'); plt.xlabel('Clase predicha')
    plt.tight_layout(); plt.savefig(outpath, dpi=150); plt.close()
    np.save(str(outpath.with_suffix('.npy')), cm)

def run_epoch(model, loader, criterion, optimizer, device, training=True):
    model.train() if training else model.eval()
    losses, preds_all, tgts_all = [], [], []
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for imgs, tgts in loader:
            imgs, tgts = imgs.to(device), tgts.to(device)
            out  = model(imgs)
            loss = criterion(out, tgts)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            losses.append(loss.item())
            preds_all.extend(out.argmax(1).cpu().numpy())
            tgts_all.extend(tgts.cpu().numpy())
    avg_loss = float(np.mean(losses))
    f1  = float(f1_score(tgts_all, preds_all, average='macro', zero_division=0))
    acc = float(accuracy_score(tgts_all, preds_all))
    return avg_loss, f1, acc, np.array(preds_all), np.array(tgts_all)

"""def train_model(model, train_loader, val_loader):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=LR,
                                momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5)"""

def train_model(model, train_loader, val_loader):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(        # ← cambio de SGD a AdamW
        model.parameters(),
        lr=1e-4,                          # ← lr=1e-4 en lugar de 0.01
        weight_decay=1e-2                 # ← wd=1e-2 explícito
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5)

    # ... resto igual

    # ... resto igual

    history = {k: [] for k in ('train_loss','val_loss','train_f1','val_f1','epoch_times')}
    best_val_acc, best_state = -1.0, None

    for epoch in range(1, EPOCHS + 1):
        t0 = time.perf_counter()
        tr_loss, tr_f1, tr_acc, _, _ = run_epoch(model, train_loader, criterion, optimizer, DEVICE, training=True)
        vl_loss, vl_f1, vl_acc, vp, vt = run_epoch(model, val_loader, criterion, None, DEVICE, training=False)
        scheduler.step(vl_loss)
        elapsed = time.perf_counter() - t0

        for k, v in zip(('train_loss','val_loss','train_f1','val_f1','epoch_times'),
                        (tr_loss, vl_loss, tr_f1, vl_f1, elapsed)):
            history[k].append(v)

        lr_now = optimizer.param_groups[0]['lr']
        print(f'Ep {epoch:3d}/{EPOCHS} | '
              f'tr_loss={tr_loss:.4f} tr_f1={tr_f1:.4f} tr_acc={tr_acc:.4f} | '
              f'vl_loss={vl_loss:.4f} vl_f1={vl_f1:.4f} vl_acc={vl_acc:.4f} | '
              f'lr={lr_now:.5f} | {elapsed:.1f}s')

        if epoch % CHECKPOINT_EVERY == 0:
            save_checkpoint({'epoch': epoch, 'model_state': model.state_dict(),
                             'optimizer_state': optimizer.state_dict(), 'history': history},
                            CHECKPOINT_DIR / f'checkpoint_epoch_{epoch:03d}.pth')

        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            best_state = {'epoch': epoch, 'val_acc': vl_acc,
                          'model_state': copy.deepcopy(model.state_dict()),
                          'history': history}
            save_checkpoint(best_state, CHECKPOINT_DIR / 'best_model.pth')
            print(f'  ★ Mejor modelo (val_acc={vl_acc:.4f})')

    save_checkpoint({'epoch': EPOCHS, 'model_state': model.state_dict(), 'history': history},
                    CHECKPOINT_DIR / 'final_model.pth')
    return history, best_state

print('Funciones definidas')

Funciones definidas


In [ ]:
# CELDA 6 — Entrenamiento y evaluación final
model    = AlexNet(in_dim=3, n_classes=n_classes).to(DEVICE)
n_params = count_parameters(model)
print(f'Parámetros: {n_params:,} ({n_params/1e6:.3f} M)')

history, best_state = train_model(model, train_loader, val_loader)

# Guardar curvas e historial
save_curves(history, PLOTS_DIR)
with open(METRICS_DIR / 'history.json', 'w') as f:
    json.dump(history, f, indent=2)

criterion = nn.CrossEntropyLoss()

# Evaluación en validación
model.load_state_dict(best_state['model_state'])
model.to(DEVICE)
_, vl_f1, vl_acc, vl_preds, vl_tgts = run_epoch(
    model, val_loader, criterion, None, DEVICE, training=False)
print(f'[VAL  best] acc={vl_acc:.4f}  f1={vl_f1:.4f}')
save_confusion_matrix(vl_tgts, vl_preds, class_names, PLOTS_DIR / 'confusion_matrix_val.png')
report = classification_report(vl_tgts, vl_preds, target_names=class_names, digits=4, zero_division=0)
(METRICS_DIR / 'classification_report_val.txt').write_text(report)
np.save(METRICS_DIR / 'val_preds.npy',   vl_preds)
np.save(METRICS_DIR / 'val_targets.npy', vl_tgts)

# Evaluación en test
_, ts_f1, ts_acc, ts_preds, ts_tgts = run_epoch(
    model, test_loader, criterion, None, DEVICE, training=False)
print(f'[TEST best] acc={ts_acc:.4f}  f1={ts_f1:.4f}')
(METRICS_DIR / 'test_metrics.txt').write_text(
    f'test_acc={ts_acc:.4f}\ntest_f1={ts_f1:.4f}\n')
np.save(METRICS_DIR / 'test_preds.npy',   ts_preds)
np.save(METRICS_DIR / 'test_targets.npy', ts_tgts)

# Resumen final
summary = (
    f'n_parameters={n_params}\n'
    f'avg_time_per_epoch={np.mean(history["epoch_times"]):.2f}s\n'
    f'best_val_acc={best_state["val_acc"]:.4f}\n'
    f'best_val_f1={max(history["val_f1"]):.4f}\n'
    f'test_acc={ts_acc:.4f}\n'
    f'test_f1={ts_f1:.4f}\n'
)
(METRICS_DIR / 'final_summary.txt').write_text(summary)
print('\n' + summary)

Parámetros: 11,359,301 (11.359 M)
Ep   1/60 | tr_loss=1.6129 tr_f1=0.2098 tr_acc=0.2131 | vl_loss=1.6082 vl_f1=0.1823 vl_acc=0.2143 | lr=0.00010 | 20.3s
  ★ Mejor modelo (val_acc=0.2143)
Ep   2/60 | tr_loss=1.6067 tr_f1=0.2040 tr_acc=0.2162 | vl_loss=1.6089 vl_f1=0.1519 vl_acc=0.2047 | lr=0.00010 | 19.8s
Ep   3/60 | tr_loss=1.6055 tr_f1=0.1980 tr_acc=0.2195 | vl_loss=1.6091 vl_f1=0.1590 vl_acc=0.2103 | lr=0.00010 | 19.9s
Ep   4/60 | tr_loss=1.6054 tr_f1=0.1924 tr_acc=0.2202 | vl_loss=1.6088 vl_f1=0.1278 vl_acc=0.2113 | lr=0.00010 | 19.9s
Ep   5/60 | tr_loss=1.6049 tr_f1=0.1866 tr_acc=0.2185 | vl_loss=1.6081 vl_f1=0.1462 vl_acc=0.2093 | lr=0.00010 | 19.8s
Ep   6/60 | tr_loss=1.6047 tr_f1=0.1829 tr_acc=0.2200 | vl_loss=1.6080 vl_f1=0.1465 vl_acc=0.2130 | lr=0.00010 | 19.8s
Ep   7/60 | tr_loss=1.6041 tr_f1=0.1770 tr_acc=0.2179 | vl_loss=1.6090 vl_f1=0.1552 vl_acc=0.2107 | lr=0.00010 | 20.0s
Ep   8/60 | tr_loss=1.6039 tr_f1=0.2011 tr_acc=0.2225 | vl_loss=1.6070 vl_f1=0.1823 vl_acc=0.2183 |